# MedNorm Learned L4 Resolver v2 Training

Status: IMPLEMENTED_UNTRAINED. Builds deterministic targets from governed train and validation data only.

In [ ]:
from pathlib import Path
import hashlib
import json
import random

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts" / "phase2_l4_learned_v2"
SMOKE_OUTPUT_DIR = ARTIFACT_ROOT / "smoke"
FULL_OUTPUT_DIR = ARTIFACT_ROOT / "full_training"
RUN_FULL_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
SEED = 20260727
MODEL_REVISION = "learned-l4-linear-v2"
OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
assert not (RUN_FULL_TRAINING and RESUME_FROM_SMOKE_CHECKPOINT)
if RUN_FULL_TRAINING:
    assert CONFIRM_FULL == "I_AUTHORIZE_L4_V2_FULL_TRAINING"


In [ ]:
EXPECTED_CORPUS_HASHES = {
    "public_ner_train.jsonl": "892dc22d7e051e05f9c96d90f42dfde7f38083a74bba6fe65b5c1d9dd05e2a4a",
    "public_ner_validation.jsonl": "ed7cdd2d49799cef0a868b6c75a3df4ca1e93ed03223337a7d31afe40f68f103",
}
CORPUS_DIR = DRIVE_ROOT / "data" / "processed"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def validate_corpus_hashes(corpus_dir: Path) -> dict[str, str]:
    observed = {}
    for name, expected in EXPECTED_CORPUS_HASHES.items():
        path = corpus_dir / name
        assert path.is_file(), f"missing governed corpus file: {path}"
        digest = sha256_file(path)
        assert digest == expected, f"hash mismatch for {name}"
        observed[name] = digest
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)


In [ ]:
import sys
from dataclasses import replace
sys.path.insert(0, str(REPO_DIR / "src"))
from mednorm_vi.lattice import build_span_lattice
from mednorm_vi.mention_factory.neural.decoding import NeuralSpan
from mednorm_vi.resolution.learned_v2 import GoldMention, ResolverV2Config, build_training_examples, grouped_train_validation_split

text = "Bệnh nhân suy tim và ho khan"
diag_start = text.index("suy tim")
sym_start = text.index("ho khan")
lattice = build_span_lattice("smoke-l4", text, neural_spans=(NeuralSpan(diag_start, diag_start + len("suy tim"), "SYMPTOM", "suy tim", 0.7, 2), NeuralSpan(sym_start, sym_start + len("ho khan"), "SYMPTOM", "ho khan", 0.9, 2)))
gold = (GoldMention("smoke-l4", diag_start, diag_start + len("suy tim"), "suy tim", "DIAGNOSIS", "source-a"), GoldMention("smoke-l4", sym_start, sym_start + len("ho khan"), "ho khan", "SYMPTOM", "source-a"))
examples_a = build_training_examples(lattice, gold, split="train", source_group="source-a", config=ResolverV2Config())
examples_b = tuple(replace(example, source_group="source-b", example_id=example.example_id + "b") for example in examples_a)
train_examples, validation_examples, split_id = grouped_train_validation_split(examples_a + examples_b, validation_fraction=0.5, seed=SEED)
assert {ex.source_group for ex in train_examples}.isdisjoint({ex.source_group for ex in validation_examples})
assert any(ex.type_target.diagnosis_symptom_pair for ex in examples_a)
preflight_report = {"training_examples": len(examples_a), "split_id": split_id, "corpus_hashes": corpus_hashes}
(OUTPUT_DIR / "preflight_report.json").write_text(json.dumps(preflight_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
from mednorm_vi.training.manifests import RunManifest

manifest = RunManifest(
    stage="smoke" if not RUN_FULL_TRAINING else "training",
    expert="learned-l4-boundary-type-resolver-v2",
    config_sha256=hashlib.sha256(b"learned-l4-boundary-type-resolver-v2").hexdigest(),
    data_sha256=hashlib.sha256(json.dumps(corpus_hashes, sort_keys=True).encode()).hexdigest(),
    corpus_sha256=corpus_hashes["public_ner_train.jsonl"],
    model_revision=MODEL_REVISION,
    seed=SEED,
    git_commit="COLAB_RESOLVES_BEFORE_FULL_TRAINING",
    checkpoint_sha256="UNAVAILABLE_UNTRAINED" if not RUN_FULL_TRAINING else "WRITTEN_AFTER_SAVE_RELOAD_VALIDATION",
    parameter_count=0,
    train_split_id="grouped_train_v2",
    validation_split_id=split_id,
    internal_test_accessed=False,
)
manifest.validate()
manifest.write_json(OUTPUT_DIR / "run_manifest.json")

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    assert path.is_file()
    assert sha256_file(path) == expected_sha256

if RUN_FULL_TRAINING:
    checkpoint_path = OUTPUT_DIR / "checkpoint" / "learned_l4_v2.json"
    validate_checkpoint_after_save_reload(checkpoint_path, manifest.checkpoint_sha256)
